In [1]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
from google.colab import files

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [2]:
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Uploaded: []


In [3]:
remaining = pd.read_csv("remaining_65_files.csv")
events_clean = pd.read_csv("events_clean.csv")

print("Remaining rows:", len(remaining))
print("events_clean rows:", len(events_clean))
display(remaining.head())
display(events_clean.head())

Remaining rows: 69
events_clean rows: 2955


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,first_400_chars,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,A. Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
1,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
2,ABT,2024-02-16,10-K,0001628280-24-005348,ABT_20240216_10-K_000162828024005348.txt,733,0.059546,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
3,APH,2021-07-30,10-Q,0001558370-21-009700,APH_20210730_10-Q_000155837021009700.txt,9495,0.037938,False,False,False,...,are specifically to our continuing operations ...,True,flagged,partial,appears to start slightly after heading/introd...,NaN,partial,appears to start slightly after heading/introd...,NaN,NaN
4,APH,2021-10-29,10-Q,0001558370-21-013825,APH_20211029_10-Q_000155837021013825.txt,9780,0.039108,False,False,False,...,are specifically to our continuing operations ...,True,flagged,partial,appears to start slightly after heading/introd...,NaN,partial,appears to start slightly after heading/introd...,NaN,NaN


,ticker,cik,filing_date,filing_type,accession_number,year,quarter
0,AAPL,320193,2019-01-30,10-Q,0000320193-19-000010,2019,1
1,AAPL,320193,2019-05-01,10-Q,0000320193-19-000066,2019,2
2,AAPL,320193,2019-07-31,10-Q,0000320193-19-000076,2019,3
3,AAPL,320193,2019-10-31,10-K,0000320193-19-000119,2019,4
4,AAPL,320193,2020-01-29,10-Q,0000320193-20-000010,2020,1


In [4]:
for df in [remaining, events_clean]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()

remaining["filing_date"] = pd.to_datetime(remaining["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")
events_clean["filing_date"] = pd.to_datetime(events_clean["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

In [5]:
repair_df = remaining.merge(
    events_clean[["ticker", "cik", "filing_date", "filing_type", "accession_number"]],
    on=["ticker", "filing_date", "filing_type", "accession_number"],
    how="left"
)

print("Rows after merge:", len(repair_df))
print("Missing cik:", repair_df["cik"].isna().sum())
display(repair_df.head())

Rows after merge: 69
Missing cik: 0


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes,cik
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
1,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1800
2,ABT,2024-02-16,10-K,0001628280-24-005348,ABT_20240216_10-K_000162828024005348.txt,733,0.059546,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1800
3,APH,2021-07-30,10-Q,0001558370-21-009700,APH_20210730_10-Q_000155837021009700.txt,9495,0.037938,False,False,False,...,True,flagged,partial,appears to start slightly after heading/introd...,NaN,partial,appears to start slightly after heading/introd...,NaN,NaN,820313
4,APH,2021-10-29,10-Q,0001558370-21-013825,APH_20211029_10-Q_000155837021013825.txt,9780,0.039108,False,False,False,...,True,flagged,partial,appears to start slightly after heading/introd...,NaN,partial,appears to start slightly after heading/introd...,NaN,NaN,820313


In [6]:
BASE_DIR = Path("/content/remaining_65_repair")
OUT_DIR = BASE_DIR / "repaired_txt"
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPAIR_LOG_CSV = BASE_DIR / "remaining_65_repair_log.csv"

In [7]:
HEADERS = {
    "User-Agent": "Your Name your_email@example.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

def cik_nolead(cik):
    if pd.isna(cik):
        return None
    return str(int(float(cik)))

def acc_nodash(acc):
    return str(acc).replace("-", "")

def filing_base_url(cik, accession_number):
    return f"https://www.sec.gov/Archives/edgar/data/{cik_nolead(cik)}/{acc_nodash(accession_number)}"

def index_json_url(cik, accession_number):
    return filing_base_url(cik, accession_number) + "/index.json"

def get_index_json(cik, accession_number):
    url = index_json_url(cik, accession_number)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def safe_int(value, default=0):
    try:
        value = str(value).strip()
        if value == "" or value.lower() == "none":
            return default
        return int(value)
    except Exception:
        return default

In [8]:
def choose_primary_doc(index_json, filing_type):
    try:
        items = index_json["directory"]["item"]
    except Exception:
        return None

    docs = []
    for x in items:
        name = str(x.get("name", ""))
        lower = name.lower()
        if lower.endswith((".htm", ".html", ".txt")):
            docs.append(x)

    if not docs:
        return None

    bad_words = ["ex-", "exhibit", "xbrl", "xml", "graphic", "image", "zip", "def14a", "8-k"]
    filtered = []
    for d in docs:
        nm = str(d.get("name", "")).lower()
        if any(b in nm for b in bad_words):
            continue
        filtered.append(d)

    if not filtered:
        filtered = docs

    ft = filing_type.lower().replace("-", "").replace("_", "")
    filing_pref = []
    for d in filtered:
        nm = str(d.get("name", "")).lower().replace("-", "").replace("_", "")
        if ft in nm:
            filing_pref.append(d)
        elif filing_type == "10-K" and "10k" in nm:
            filing_pref.append(d)
        elif filing_type == "10-Q" and "10q" in nm:
            filing_pref.append(d)

    if filing_pref:
        filtered = filing_pref

    filtered = sorted(filtered, key=lambda x: safe_int(x.get("size", 0), 0), reverse=True)
    return filtered[0]["name"] if filtered else None

In [9]:
def download_filing_doc(cik, accession_number, filename):
    url = filing_base_url(cik, accession_number) + f"/{filename}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def html_to_text(html):
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "ix:header", "header", "footer"]):
        tag.decompose()

    text = soup.get_text("\n")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

def normalize_text(text):
    text = text.replace("\xa0", " ")
    text = text.replace("’", "'")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text

In [10]:
def is_toc_like(chunk):
    c = chunk.lower()
    return (
        "table of contents" in c or
        "under the heading" in c or
        "refer to item" in c or
        "see item" in c or
        ("item 7a" in c and "item 8" in c) or
        ("item 3" in c and "item 4" in c)
    )

def get_lines_with_offsets(text):
    lines = []
    pos = 0
    for line in text.splitlines(True):
        raw = line
        clean = re.sub(r"\s+", " ", raw).strip()
        lines.append((pos, raw, clean))
        pos += len(raw)
    return lines

def find_start_candidates(lines, filing_type):
    candidates = []

    if filing_type == "10-K":
        item_num = "7"
    else:
        item_num = "2"

    for i in range(len(lines)):
        clean = lines[i][2].lower().replace("’", "'")
        next_clean = lines[i+1][2].lower().replace("’", "'") if i + 1 < len(lines) else ""
        next2_clean = lines[i+2][2].lower().replace("’", "'") if i + 2 < len(lines) else ""

        combined2 = f"{clean} {next_clean}".strip()
        combined3 = f"{clean} {next_clean} {next2_clean}".strip()

        patterns = [
            rf"^item\s*{item_num}[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            rf"^item\s*{item_num}[\.\-:\s]+management[' ]s discussion and analysis",
            rf"^item\s*{item_num}\b",
        ]

        match = False
        for pat in patterns:
            if re.search(pat, clean, flags=re.I):
                match = True
            if re.search(pat, combined2, flags=re.I):
                match = True
            if re.search(pat, combined3, flags=re.I):
                match = True

        if match:
            chunk = " ".join(x[2] for x in lines[i:i+8]).lower()
            if is_toc_like(chunk):
                continue
            if "part i item 1 business" in chunk:
                continue
            candidates.append(lines[i][0])

    return sorted(set(candidates))

def find_end_candidates(lines, filing_type):
    candidates = []

    if filing_type == "10-K":
        end_pats = [
            r"^item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
            r"^item\s*8[\.\-:\s]+financial statements",
            r"^item\s*8\b",
        ]
    else:
        end_pats = [
            r"^item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
            r"^item\s*4[\.\-:\s]+controls and procedures",
            r"^item\s*3\b",
            r"^item\s*4\b",
        ]

    for i in range(len(lines)):
        clean = lines[i][2].lower().replace("’", "'")
        next_clean = lines[i+1][2].lower().replace("’", "'") if i + 1 < len(lines) else ""
        combined2 = f"{clean} {next_clean}".strip()

        for pat in end_pats:
            if re.search(pat, clean, flags=re.I) or re.search(pat, combined2, flags=re.I):
                candidates.append(lines[i][0])
                break

    return sorted(set(candidates))

In [11]:
def choose_best_span(text, filing_type):
    lines = get_lines_with_offsets(text)
    starts = find_start_candidates(lines, filing_type)
    ends = find_end_candidates(lines, filing_type)

    if not starts:
        return None, None, "NO_START"
    if not ends:
        return None, None, "NO_END"

    doc_len = len(text)
    best = None
    best_score = -10**9

    for s in starts:
        valid_ends = [e for e in ends if e > s]
        if not valid_ends:
            continue

        e = valid_ends[0]
        span = e - s
        if span < 3000 or span > 400000:
            continue

        start_chunk = text[max(0, s-300): min(doc_len, s+1000)].lower()
        score = 0

        if is_toc_like(start_chunk):
            score -= 1000
        if "part i item 1 business" in start_chunk:
            score -= 1200

        score += (s / max(doc_len, 1)) * 200
        score += -abs(span - 50000) / 50000 * 50

        if score > best_score:
            best_score = score
            best = (s, e)

    if best is None:
        return None, None, "NO_VALID_SPAN"

    return best[0], best[1], "OK"

In [12]:
def extract_mda_stronger(text, filing_type):
    text = normalize_text(text)
    s, e, status = choose_best_span(text, filing_type)

    if status != "OK":
        return None, status

    mda = text[s:e].strip()
    wc = len(re.findall(r"\b[a-zA-Z]+\b", mda))
    if wc < 800:
        return None, "TOO_SHORT"

    first_400 = re.sub(r"\s+", " ", mda[:400]).lower().replace("’", "'")

    if filing_type == "10-K":
        good = ("item 7" in first_400 and "management's discussion" in first_400)
    else:
        good = ("item 2" in first_400 and "management's discussion" in first_400)

    if not good:
        return None, "BAD_START"

    if is_toc_like(first_400):
        return None, "BAD_START"

    return mda, "OK"

In [13]:
def repair_one_filing_stronger(row, sleep_seconds=0.2):
    ticker = row["ticker"]
    cik = row["cik"]
    filing_date = row["filing_date"]
    filing_type = row["filing_type"]
    accession_number = row["accession_number"]

    out = {
        "ticker": ticker,
        "filing_date": filing_date,
        "filing_type": filing_type,
        "accession_number": accession_number,
        "original_manual_label": row.get("manual_label", ""),
        "repair_status": "",
        "primary_doc": "",
        "repaired_filename": ""
    }

    try:
        if pd.isna(cik):
            out["repair_status"] = "NO_CIK"
            return out

        idx = get_index_json(cik, accession_number)
        primary_doc = choose_primary_doc(idx, filing_type)

        if primary_doc is None:
            out["repair_status"] = "NO_PRIMARY_DOC"
            return out

        out["primary_doc"] = primary_doc

        html = download_filing_doc(cik, accession_number, primary_doc)
        text = html_to_text(html)

        mda_text, status = extract_mda_stronger(text, filing_type)

        if status != "OK":
            out["repair_status"] = status
            return out

        repaired_filename = f"{ticker}_{filing_date}_{filing_type}_{accession_number}_STRONGER65.txt"
        repaired_path = OUT_DIR / repaired_filename

        with open(repaired_path, "w", encoding="utf-8") as f:
            f.write(mda_text)

        out["repair_status"] = "OK"
        out["repaired_filename"] = repaired_filename

        time.sleep(sleep_seconds)
        return out

    except Exception as e:
        out["repair_status"] = f"ERROR: {str(e)[:120]}"
        return out

In [14]:
results = []

for i, row in repair_df.iterrows():
    result = repair_one_filing_stronger(row)
    results.append(result)

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1}/{len(repair_df)}")

repair_log = pd.DataFrame(results)
repair_log.to_csv(REPAIR_LOG_CSV, index=False)

print("Repair complete.")
display(repair_log["repair_status"].value_counts())
display(repair_log.head())

Processed 10/69
Processed 20/69
Processed 30/69
Processed 40/69
Processed 50/69
Processed 60/69
Repair complete.


,count
repair_status,
NO_START,36
OK,17
NO_END,14
BAD_START,2


,ticker,filing_date,filing_type,accession_number,original_manual_label,repair_status,primary_doc,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,incorrect,NO_START,a10-k20199282019.htm,
1,ABT,2023-02-17,10-K,0001628280-23-004026,incorrect,NO_START,abt-20221231x10kexx21.htm,
2,ABT,2024-02-16,10-K,0001628280-24-005348,incorrect,NO_START,abt-20231231x10kexx312.htm,
3,APH,2021-07-30,10-Q,0001558370-21-009700,partial,OK,aph-20210630x10q.htm,APH_2021-07-30_10-Q_0001558370-21-009700_STRON...
4,APH,2021-10-29,10-Q,0001558370-21-013825,partial,OK,aph-20210930x10q.htm,APH_2021-10-29_10-Q_0001558370-21-013825_STRON...


In [15]:
ok_df = repair_log.loc[
    repair_log["repair_status"] == "OK",
    ["ticker", "filing_date", "filing_type", "accession_number", "repaired_filename"]
].dropna().copy()

print("Recovered OK files:", len(ok_df))
display(ok_df.head())

Recovered OK files: 17


,ticker,filing_date,filing_type,accession_number,repaired_filename
3,APH,2021-07-30,10-Q,0001558370-21-009700,APH_2021-07-30_10-Q_0001558370-21-009700_STRON...
4,APH,2021-10-29,10-Q,0001558370-21-013825,APH_2021-10-29_10-Q_0001558370-21-013825_STRON...
5,APH,2022-07-29,10-Q,0001558370-22-011379,APH_2022-07-29_10-Q_0001558370-22-011379_STRON...
6,APH,2022-10-28,10-Q,0001558370-22-015585,APH_2022-10-28_10-Q_0001558370-22-015585_STRON...
7,CB,2019-02-28,10-K,0000896159-19-000005,CB_2019-02-28_10-K_0000896159-19-000005_STRONG...


In [16]:
def read_text_safe(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def first_chunk(text, n=1500):
    return re.sub(r"\s+", " ", text[:n]).strip()

def word_count_alpha(text):
    return len(re.findall(r"\b[a-zA-Z]+\b", text))

def validate_start_strict(text, filing_type):
    t = first_chunk(text, 1500).lower().replace("’", "'")

    if filing_type == "10-K":
        strong_heading = bool(re.search(
            r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis",
            t
        ))
    else:
        strong_heading = bool(re.search(
            r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis",
            t
        ))

    bad_reference = (
        "under the heading" in t or
        "see item" in t or
        "refer to item" in t or
        "table of contents" in t or
        "part i item 1" in t or
        "company background" in t or
        ("item 7a" in t[:500] and "item 8" in t[:500]) or
        ("item 3" in t[:500] and "item 4" in t[:500])
    )

    return {
        "strong_heading_match": strong_heading,
        "bad_reference_start": bad_reference,
        "first_300_chars": text[:300].replace("\n", " "),
        "word_count": word_count_alpha(text)
    }

In [17]:
records = []

for _, row in ok_df.iterrows():
    path = OUT_DIR / row["repaired_filename"]
    text = read_text_safe(path)
    checks = validate_start_strict(text, row["filing_type"])

    records.append({
        "ticker": row["ticker"],
        "filing_date": row["filing_date"],
        "filing_type": row["filing_type"],
        "accession_number": row["accession_number"],
        "repaired_filename": row["repaired_filename"],
        **checks
    })

ok_validation_df = pd.DataFrame(records)
display(ok_validation_df)

,ticker,filing_date,filing_type,accession_number,repaired_filename,strong_heading_match,bad_reference_start,first_300_chars,word_count
0,APH,2021-07-30,10-Q,0001558370-21-009700,APH_2021-07-30_10-Q_0001558370-21-009700_STRON...,True,False,Item 2. MANAGEMENT'S DISCUSSION AND ANALYSIS ​...,9608
1,APH,2021-10-29,10-Q,0001558370-21-013825,APH_2021-10-29_10-Q_0001558370-21-013825_STRON...,True,False,Item 2. MANAGEMENT'S DISCUSSION AND ANALYSIS ​...,9895
2,APH,2022-07-29,10-Q,0001558370-22-011379,APH_2022-07-29_10-Q_0001558370-22-011379_STRON...,True,False,Item 2. MANAGEMENT'S DISCUSSION AND ANALYSIS ​...,10295
3,APH,2022-10-28,10-Q,0001558370-22-015585,APH_2022-10-28_10-Q_0001558370-22-015585_STRON...,True,False,Item 2. MANAGEMENT'S DISCUSSION AND ANALYSIS ​...,10882
4,CB,2019-02-28,10-K,0000896159-19-000005,CB_2019-02-28_10-K_0000896159-19-000005_STRONG...,True,True,ITEM 7. Management's Discussion and Analysis o...,31346
5,CB,2020-02-27,10-K,0000896159-20-000003,CB_2020-02-27_10-K_0000896159-20-000003_STRONG...,True,True,ITEM 7. Management's Discussion and Analysis o...,29291
6,CMCSA,2019-01-31,10-K,0001166691-19-000005,CMCSA_2019-01-31_10-K_0001166691-19-000005_STR...,True,False,Item 7: Management's Discussion and Analysis o...,13921
7,CMCSA,2020-01-30,10-K,0001166691-20-000008,CMCSA_2020-01-30_10-K_0001166691-20-000008_STR...,True,True,Item 7: Management's Discussion and Analysis o...,13692
8,CMCSA,2021-02-04,10-K,0001166691-21-000008,CMCSA_2021-02-04_10-K_0001166691-21-000008_STR...,True,False,Item 7: Management's Discussion and Analysis o...,24444
9,GD,2020-02-10,10-K,0000040533-20-000015,GD_2020-02-10_10-K_0000040533-20-000015_STRONG...,True,False,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS ...,8961


In [18]:
ok_validation_df["needs_manual_review"] = (
    (~ok_validation_df["strong_heading_match"]) |
    (ok_validation_df["bad_reference_start"])
)

likely_good = ok_validation_df.loc[~ok_validation_df["needs_manual_review"]].copy()
needs_review = ok_validation_df.loc[ok_validation_df["needs_manual_review"]].copy()

print("Likely good:", len(likely_good))
print("Needs manual review:", len(needs_review))

Likely good: 14
Needs manual review: 3


In [20]:
def preview_repaired_file(filename, start_chars=2500, end_chars=1200):
    path = OUT_DIR / filename
    if not path.exists():
        print("File not found:", path)
        return

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print("=" * 120)
    print("FILE:", filename)
    print("=" * 120)
    print("\n--- START PREVIEW ---\n")
    print(text[:start_chars])
    print("\n--- END PREVIEW ---\n")
    print(text[-end_chars:])
    print("\nWord count:", len(re.findall(r"\b[a-zA-Z]+\b", text)))
    print("=" * 120)

In [21]:
display(needs_review[["ticker","filing_date","filing_type","repaired_filename","first_300_chars"]])

,ticker,filing_date,filing_type,repaired_filename,first_300_chars
4,CB,2019-02-28,10-K,CB_2019-02-28_10-K_0000896159-19-000005_STRONG...,ITEM 7. Management's Discussion and Analysis o...
5,CB,2020-02-27,10-K,CB_2020-02-27_10-K_0000896159-20-000003_STRONG...,ITEM 7. Management's Discussion and Analysis o...
7,CMCSA,2020-01-30,10-K,CMCSA_2020-01-30_10-K_0001166691-20-000008_STR...,Item 7: Management's Discussion and Analysis o...


In [22]:
likely_good["manual_label"] = ""
likely_good["notes"] = ""

needs_review["manual_label"] = ""
needs_review["notes"] = ""

LIKELY_GOOD_CSV = "/content/remaining65_likely_good_14.csv"
NEEDS_REVIEW_CSV = "/content/remaining65_needs_manual_review_3.csv"

likely_good.to_csv(LIKELY_GOOD_CSV, index=False)
needs_review.to_csv(NEEDS_REVIEW_CSV, index=False)

print("Saved:", LIKELY_GOOD_CSV, "rows =", len(likely_good))
print("Saved:", NEEDS_REVIEW_CSV, "rows =", len(needs_review))

Saved: /content/remaining65_likely_good_14.csv rows = 14
Saved: /content/remaining65_needs_manual_review_3.csv rows = 3


In [23]:
from google.colab import files

files.download(LIKELY_GOOD_CSV)
files.download(NEEDS_REVIEW_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
remaining65_likely = pd.read_csv("remaining65_likely_good_14.csv")
remaining65_review3 = pd.read_csv("remaining65_needs_manual_review_3.csv")

for df in [remaining65_likely, remaining65_review3]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    df["manual_label"] = df.get("manual_label", "").fillna("").astype(str).str.strip().str.lower()
    df["notes"] = df.get("notes", "").fillna("").astype(str)

correct_keys_17 = {
    ("APH", "2021-07-30", "10-Q"),
    ("APH", "2021-10-29", "10-Q"),
    ("APH", "2022-07-29", "10-Q"),
    ("APH", "2022-10-28", "10-Q"),
    ("CMCSA", "2019-01-31", "10-K"),
    ("CMCSA", "2021-02-04", "10-K"),
    ("GD", "2020-02-10", "10-K"),
    ("GD", "2021-02-09", "10-K"),
    ("MNST", "2019-02-28", "10-K"),
    ("MNST", "2020-02-28", "10-K"),
    ("MNST", "2021-03-01", "10-K"),
    ("ORCL", "2022-06-21", "10-K"),
    ("PANW", "2019-05-30", "10-Q"),
    ("PH", "2020-08-26", "10-K"),
    ("CB", "2019-02-28", "10-K"),
    ("CB", "2020-02-27", "10-K"),
    ("CMCSA", "2020-01-30", "10-K"),
}

for df in [remaining65_likely, remaining65_review3]:
    df.loc[
        df.apply(lambda r: (r["ticker"], r["filing_date"], r["filing_type"]) in correct_keys_17, axis=1),
        ["manual_label", "notes"]
    ] = ["correct", "accepted after strict review"]

correct_17 = pd.concat([remaining65_likely, remaining65_review3], ignore_index=True)
correct_17 = correct_17.loc[correct_17["manual_label"] == "correct"].copy()

print("Correct rows:", len(correct_17))
display(correct_17[["ticker","filing_date","filing_type","accession_number","repaired_filename"]])

Correct rows: 17


,ticker,filing_date,filing_type,accession_number,repaired_filename
0,APH,2021-07-30,10-Q,0001558370-21-009700,APH_2021-07-30_10-Q_0001558370-21-009700_STRON...
1,APH,2021-10-29,10-Q,0001558370-21-013825,APH_2021-10-29_10-Q_0001558370-21-013825_STRON...
2,APH,2022-07-29,10-Q,0001558370-22-011379,APH_2022-07-29_10-Q_0001558370-22-011379_STRON...
3,APH,2022-10-28,10-Q,0001558370-22-015585,APH_2022-10-28_10-Q_0001558370-22-015585_STRON...
4,CMCSA,2019-01-31,10-K,0001166691-19-000005,CMCSA_2019-01-31_10-K_0001166691-19-000005_STR...
5,CMCSA,2021-02-04,10-K,0001166691-21-000008,CMCSA_2021-02-04_10-K_0001166691-21-000008_STR...
6,GD,2020-02-10,10-K,0000040533-20-000015,GD_2020-02-10_10-K_0000040533-20-000015_STRONG...
7,GD,2021-02-09,10-K,0000040533-21-000010,GD_2021-02-09_10-K_0000040533-21-000010_STRONG...
8,MNST,2019-02-28,10-K,0001104659-19-011581,MNST_2019-02-28_10-K_0001104659-19-011581_STRO...
9,MNST,2020-02-28,10-K,0001104659-20-027209,MNST_2020-02-28_10-K_0001104659-20-027209_STRO...


In [25]:
CORRECT_17_CSV = "/content/correct_17_files.csv"
correct_17.to_csv(CORRECT_17_CSV, index=False)
print("Saved:", CORRECT_17_CSV)

Saved: /content/correct_17_files.csv


In [26]:
from pathlib import Path
import shutil
import zipfile
from google.colab import files

OUT_DIR = Path("/content/remaining_65_repair/repaired_txt")
DOWNLOAD_DIR = Path("/content/correct_17_txt")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# clear old files if any
for old_file in DOWNLOAD_DIR.glob("*.txt"):
    old_file.unlink()

copied = 0
missing = []

for _, row in correct_17.iterrows():
    fname = row["repaired_filename"]
    src = OUT_DIR / fname
    dst = DOWNLOAD_DIR / fname

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(fname)

print("Copied txt files:", copied)
print("Missing txt files:", len(missing))
if missing:
    print("Missing list:")
    for x in missing:
        print(x)

Copied txt files: 17
Missing txt files: 0


In [27]:
CORRECT_17_ZIP = "/content/correct_17_txt.zip"

with zipfile.ZipFile(CORRECT_17_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in DOWNLOAD_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

print("Saved zip:", CORRECT_17_ZIP)

Saved zip: /content/correct_17_txt.zip


In [28]:
files.download(CORRECT_17_CSV)
files.download(CORRECT_17_ZIP)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
remaining_65 = pd.read_csv("remaining_65_files.csv")

remaining_65["ticker"] = remaining_65["ticker"].astype(str).str.strip().str.upper()
remaining_65["filing_type"] = remaining_65["filing_type"].astype(str).str.strip().str.upper()
remaining_65["accession_number"] = remaining_65["accession_number"].astype(str).str.strip()
remaining_65["filing_date"] = pd.to_datetime(remaining_65["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

correct_17_keys = set(
    zip(
        correct_17["ticker"],
        correct_17["filing_date"],
        correct_17["filing_type"],
        correct_17["accession_number"]
    )
)

remaining_48 = remaining_65.loc[
    ~remaining_65.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in correct_17_keys,
        axis=1
    )
].copy()

REMAINING_48_CSV = "/content/remaining_48_files.csv"
remaining_48.to_csv(REMAINING_48_CSV, index=False)

print("Remaining unresolved files:", len(remaining_48))
print("Saved:", REMAINING_48_CSV)

Remaining unresolved files: 52
Saved: /content/remaining_48_files.csv


In [30]:
files.download(REMAINING_48_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>